# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType, DecimalType

In [0]:
RENAMED_COLUMN = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date",
}

# Read from Bronze Table

In [0]:
df = spark.table("bike_data.bronze.prd_info_raw")

In [0]:
df.display()

# Transformations

## Trimming

In [0]:
for item in df.schema.fields:
    if isinstance(item.dataType, StringType):
        df = df.withColumn(item.name, trim(col(item.name)))

## Normalization

In [0]:
df =(
    df.
    withColumn(
        "prd_cost", 
        col("prd_cost").cast("Decimal(10,2)")
    )
)


In [0]:
df = (
    df.withColumn(
        "prd_line",
        F.when(col("prd_line") == "R", "road")
        .when(col("prd_line") == "M", "mountain")
        .when(col("prd_line") == "T", "touring")
        .when(col("prd_line") == "S", "sport")
        .otherwise(col("prd_line"))
    )
)

## Rename Columns

In [0]:
for old_column, new_columnns in RENAMED_COLUMN.items():
    df = df.withColumnRenamed(old_column, new_columnns)

In [0]:
df.limit(5).display()

# Write to Silver Table

In [0]:
(df.write
 .mode("overwrite")
 .saveAsTable("bike_data.silver.crm_product_information"))

In [0]:
df.display()